In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import sys, os
sys.path.append(os.path.join(os.getcwd(), '../../')) # Add root of repo to import MBM
from typing import Optional, Iterable, Dict, List
import os
import json
import pandas as pd
import numpy as np
import warnings
import re
import matplotlib.pyplot as plt
import seaborn as sns
from cmcrameri import cm
import massbalancemachine as mbm
import logging
import torch.nn as nn
from skorch.helper import SliceDataset
from datetime import datetime
from skorch.callbacks import EarlyStopping, LRScheduler, Checkpoint
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import Dataset
import pickle 
from scipy import stats
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import torch 
from matplotlib.lines import Line2D

from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

# Initialize logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(torch.cuda.is_available())   # True
print(torch.cuda.device_count())   # 0  ← wrong$

if torch.cuda.device_count()==0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")
    
BASE_DIR = Path(cfg.dataPath) / path_cache / f"TF_REGION"
CACHE_DIR = BASE_DIR / "LSTM_cache_TL/Transformer_exp"

# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)

# make sure BASE_DIR exists
os.makedirs(CACHE_DIR, exist_ok=True)

print(f"Base directory for data: {BASE_DIR}")
print(f"Cache directory for models: {CACHE_DIR}")

In [ ]:
MONTHLY_COLS = [
    't2m',
    'tp',
    'slhf',
    'sshf',
    'ssrd',
    'fal',
    'str',
    'ELEVATION_DIFFERENCE',
]
STATIC_COLS = ['aspect', 'slope', 'svf']

feature_columns = MONTHLY_COLS + STATIC_COLS

# Transform data to monthly format (run or load data):
paths = {
    'era5_climate_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_EU_US_CANADA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_EU_US_CANADA.nc"),
}
# Check that all these files exists
for key, path in paths.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

# Transform data to monthly format (run or load data):
paths_HMA = {
    "era5_climate_data":
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
    'geopotential_data':
    os.path.join(cfg.dataPath, path_ERA5_raw,
                 "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
}
# Check that all these files exists
for key, path in paths_HMA.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Required file for {key} not found at {path}")

vois_climate = [
    "t2m",
    "tp",
    "slhf",
    "sshf",
    "ssrd",
    "fal",
    "str",
]

vois_topographical = ["aspect", "slope", "svf"]

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

### Monthly datasets:

In [ ]:
experiment_region_groups = {
    # "CEU": ["FR", "IT_AT", "CH"],
    "HMA": ["CENTRALASIA", "SOUTHASIAWEST", "SOUTHASIAEAST"]
}
paths_multi = {
    "EU_US_CANADA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_EU_US_CANADA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_EU_US_CANADA.nc"),
    },
    "HMA": {
        "era5_climate_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_monthly_averaged_data_HIGH_MOUNTAIN_ASIA.nc"),
        "geopotential_data":
        os.path.join(cfg.dataPath, path_ERA5_raw,
                     "era5_geopotential_pressure_HIGH_MOUNTAIN_ASIA.nc"),
    },
}

# Only the three regions we care about
SOURCE_REGIONS = [
    'CH', 'NOR', 'ISL', 'ALA', 'SJM', 'CENTRALASIA', 'FR', 'IT_AT'
]

# Step 1: only build monthly cache for these three
run_flag_by_code = {
    'CH': False,
    'ISL': False,
    'NOR': False,
    'ALA': False,
    'SJM': False,
    'CENTRALASIA': False,
    'FR': False,
    'IT_AT': False
}

monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=SOURCE_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

# Step 2: no XREG_PAIRS needed — just get the train/test split per region
# res_xreg_by_source[region] gives you df_train, df_test, df_train_aug, df_test_aug
res_xreg_by_source = {
    region: monthly_cache[region]  # already has the train/test split
    for region in SOURCE_REGIONS
}

## Datasets:

### Ft and pool glaciers:

In [ ]:
save_path = BASE_DIR / "glacier_splits.json"

with open(save_path, "r") as f:
    output = json.load(f)

splits = {
    src_region: {
        "pool_glaciers": data["holdout_glaciers"],  # swapped
        "holdout_glaciers": data["pool_glaciers"],  # swapped
        "actual_pool_frac":
        1.0 - data["actual_pool_frac"],  # update fraction too
        "sinkhorn_pool_vs_holdout": data["sinkhorn_pool_vs_holdout"],
        "sinkhorn_pool_vs_region":
        data["sinkhorn_holdout_vs_region"],  # swapped
        "sinkhorn_holdout_vs_region":
        data["sinkhorn_pool_vs_region"],  # swapped
        "blur_joint": data["blur_joint"],
        "best_seed": data["best_seed"],
        "best_score": data["best_score"],
    }
    for src_region, data in output.items()
}

print(f"Loaded splits from: {save_path}")
for src_region, split in splits.items():
    print(f"\n{src_region}:")
    print(f"  Pool glaciers     : {split['pool_glaciers']}")
    print(f"  Holdout glaciers  : {split['holdout_glaciers']}")
    print(f"  Pool fraction     : {split['actual_pool_frac']:.1%}")
    print(f"  Sk(pool↔holdout)  : {split['sinkhorn_pool_vs_holdout']:.4f}")

### Sequence datasets:

In [ ]:
def build_pooled_assets_from_cache(
    res_xreg_by_source: dict,
    splits: dict,
    source_regions: list,
    monthly_cols: list,
    static_cols: list,
    cfg,
    seed: int = 42,
    val_ratio: float = 0.2,
    cache_dir: str = "logs/LSTM_cache_pooled",
    force_recompute: bool = False,
):
    import joblib
    os.makedirs(cache_dir, exist_ok=True)

    # cache key encodes which regions are pooled
    regions_key = "_".join(sorted(source_regions))
    train_p = os.path.join(cache_dir, f"pooled_{regions_key}_train.joblib")
    split_p = os.path.join(cache_dir, f"pooled_{regions_key}_split.joblib")

    # one cache file per holdout region
    holdout_ps = {
        region: os.path.join(cache_dir, f"holdout_{region}.joblib")
        for region in source_regions
    }

    all_cached = (os.path.exists(train_p) and os.path.exists(split_p)
                  and all(os.path.exists(p) for p in holdout_ps.values()))

    if all_cached and not force_recompute:
        print(f"Loading pooled assets from cache: {cache_dir}")
        ds_pooled = joblib.load(train_p)
        split = joblib.load(split_p)
        holdout_datasets = {
            region: joblib.load(p)
            for region, p in holdout_ps.items()
        }
        print(
            f"Pooled dataset: {len(ds_pooled)} sequences "
            f"({len(split['train_idx'])} train | {len(split['val_idx'])} val)")
        for region, ds_h in holdout_datasets.items():
            print(f"  {region} holdout: {len(ds_h)} sequences")
        return (
            {
                "ds_train": ds_pooled,
                "train_idx": split["train_idx"],
                "val_idx": split["val_idx"]
            },
            holdout_datasets,
        )

    # --- build fresh ---
    print(f"Building pooled assets for regions: {source_regions}")
    train_dfs_loss, train_dfs_full = [], []
    holdout_datasets = {}
    months_head_pad = months_tail_pad = None

    for region in source_regions:
        df_loss = res_xreg_by_source[region]['data_monthly']
        df_full = res_xreg_by_source[region]['data_monthly_aug']
        months_head_pad = res_xreg_by_source[region]['months_head_pad']
        months_tail_pad = res_xreg_by_source[region]['months_tail_pad']

        holdout_glaciers = splits[region]['holdout_glaciers']

        df_loss_train = df_loss[~df_loss['GLACIER'].isin(holdout_glaciers)]
        df_full_train = df_full[~df_full['GLACIER'].isin(holdout_glaciers)]
        df_loss_hold = df_loss[df_loss['GLACIER'].isin(holdout_glaciers)]
        df_full_hold = df_full[df_full['GLACIER'].isin(holdout_glaciers)]

        n_total = df_loss['GLACIER'].nunique()
        n_train = df_loss_train['GLACIER'].nunique()
        n_hold = df_loss_hold['GLACIER'].nunique()
        print(
            f"{region}: {n_total} glaciers → {n_train} train | {n_hold} holdout"
        )

        train_dfs_loss.append(df_loss_train)
        train_dfs_full.append(df_full_train)

        ds_holdout = build_combined_LSTM_dataset(
            df_loss=df_loss_hold,
            df_full=df_full_hold,
            monthly_cols=monthly_cols,
            static_cols=static_cols,
            months_head_pad=months_head_pad,
            months_tail_pad=months_tail_pad,
            normalize_target=True,
            expect_target=True,
            show_progress=False,
        )
        holdout_datasets[region] = ds_holdout
        joblib.dump(ds_holdout, holdout_ps[region], compress=3)
        print(f"  Saved {region} holdout -> {holdout_ps[region]}")

    # --- pooled train ---
    df_pooled_loss = pd.concat(train_dfs_loss, ignore_index=True)
    df_pooled_full = pd.concat(train_dfs_full, ignore_index=True)

    print(f"\nPooled head pad: {months_head_pad}")
    print(f"Pooled tail pad: {months_tail_pad}")

    mbm.utils.seed_all(seed)

    ds_pooled = build_combined_LSTM_dataset(
        df_loss=df_pooled_loss,
        df_full=df_pooled_full,
        monthly_cols=monthly_cols,
        static_cols=static_cols,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        normalize_target=True,
        expect_target=True,
        show_progress=True,
    )

    train_idx, val_idx = mbm.data_processing.MBSequenceDataset.split_indices(
        len(ds_pooled), val_ratio=val_ratio, seed=seed)

    # --- save ---
    joblib.dump(ds_pooled, train_p, compress=3)
    joblib.dump({
        "train_idx": train_idx,
        "val_idx": val_idx
    },
                split_p,
                compress=3)
    print(f"\nSaved pooled train  -> {train_p}")
    print(f"Saved split indices -> {split_p}")
    print(f"Pooled dataset: {len(ds_pooled)} sequences "
          f"({len(train_idx)} train | {len(val_idx)} val)")

    return (
        {
            "ds_train": ds_pooled,
            "train_idx": train_idx,
            "val_idx": val_idx
        },
        holdout_datasets,
    )

In [ ]:
pooled_assets, ds_holdout_by_region = build_pooled_assets_from_cache(
    res_xreg_by_source=res_xreg_by_source,
    splits=splits,
    source_regions=SOURCE_REGIONS,  # ['CH', 'NOR', 'ISL']
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    seed=cfg.seed,
    val_ratio=0.2,
    cache_dir=CACHE_DIR,
    force_recompute=True,
)

In [ ]:
ds_pooled = pooled_assets["ds_train"]
train_idx = pooled_assets["train_idx"]
val_idx = pooled_assets["val_idx"]

# ---- 1. Sizes ----
print("=== Sizes ===")
print(f"Pooled dataset total sequences : {len(ds_pooled)}")
print(f"Train split                    : {len(train_idx)}")
print(f"Val split                      : {len(val_idx)}")
print(
    f"Sum matches total              : {len(train_idx) + len(val_idx) == len(ds_pooled)}"
)

# ---- 2. Tensor shapes ----
print("\n=== Tensor shapes ===")
print(
    f"Xm (monthly inputs) : {tuple(ds_pooled.Xm.shape)}  expected (N, T, {len(MONTHLY_COLS)})"
)
print(
    f"Xs (static inputs)  : {tuple(ds_pooled.Xs.shape)}  expected (N, {len(STATIC_COLS)})"
)
print(f"mv (valid mask)     : {tuple(ds_pooled.mv.shape)}")
print(f"y  (targets)        : {tuple(ds_pooled.y.shape)}")

# ---- 3. Pristine check (no scalers fitted yet) ----
print("\n=== Scaler state (should all be None) ===")
print(f"month_mean  : {ds_pooled.month_mean}")
print(f"static_mean : {ds_pooled.static_mean}")
print(f"y_mean      : {ds_pooled.y_mean}")

# ---- 4. NaN / Inf check ----
print("\n=== NaN / Inf check ===")
for name, t in [("Xm", ds_pooled.Xm), ("Xs", ds_pooled.Xs),
                ("y", ds_pooled.y)]:
    n_nan = torch.isnan(t).sum().item()
    n_inf = torch.isinf(t).sum().item()
    status = "OK" if (n_nan == 0 and n_inf == 0) else "ISSUE"
    print(f"  {name}: NaNs={n_nan}, Infs={n_inf}  [{status}]")

# ---- 5. Region coverage — check all three regions appear in keys ----
print("\n=== Region coverage in keys ===")
glaciers_in_pooled = {k[0] for k in ds_pooled.keys}
for region in SOURCE_REGIONS:
    df_loss = res_xreg_by_source[region]['data_monthly']
    train_glaciers = set(df_loss['GLACIER'].unique()) - set(
        splits[region]['holdout_glaciers'])
    overlap = glaciers_in_pooled & train_glaciers
    print(
        f"  {region}: {len(overlap)} / {len(train_glaciers)} train glaciers present in pooled dataset"
    )

# ---- 6. Holdout check — holdout glaciers must NOT appear in pooled ----
print("\n=== Holdout isolation check ===")
for region in SOURCE_REGIONS:
    holdout_gls = set(splits[region]['holdout_glaciers'])
    leaked = glaciers_in_pooled & holdout_gls
    status = "OK" if len(leaked) == 0 else f"LEAK: {leaked}"
    print(f"  {region}: {status}")

# ---- 7. Holdout datasets ----
print("\n=== Holdout datasets ===")
for region, ds_h in ds_holdout_by_region.items():
    n_w = ds_h.iw.sum().item()
    n_a = ds_h.ia.sum().item()
    holdout_gls = {k[0] for k in ds_h.keys}
    print(f"  {region}: {len(ds_h)} sequences ({n_w} winter | {n_a} annual) | "
          f"{len(holdout_gls)} glaciers")

# ---- 8. Winter / annual balance in pooled ----
print("\n=== Winter / Annual balance in pooled dataset ===")
n_w = ds_pooled.iw.sum().item()
n_a = ds_pooled.ia.sum().item()
print(f"  Winter : {n_w} ({100*n_w/len(ds_pooled):.1f}%)")
print(f"  Annual : {n_a} ({100*n_a/len(ds_pooled):.1f}%)")

## Train model:

In [ ]:
def train_or_load_one_source_model(
        cfg,
        key: str,
        lstm_assets: dict,
        best_params: dict,
        device,
        models_dir="models",
        prefix="lstm_xreg",
        train_flag=True,
        force_retrain=True,
        epochs=150,
        batch_size_train=64,
        batch_size_val=128,
        verbose=True,
        date=None,
        model_type="lstm",  # "lstm" or "transformer"
):
    """Train or load a single source-region model. No test set needed."""
    assert model_type in ("lstm", "transformer"), \
        f"model_type must be 'lstm' or 'transformer', got '{model_type}'"

    # ---- resolve model class once ----
    model_cls = mbm.models.LSTM_MB if model_type == "lstm" else mbm.models.Transformer_MB

    run_date = datetime.now().strftime("%Y-%m-%d") if date is None else date
    os.makedirs(models_dir, exist_ok=True)
    model_filename = os.path.join(models_dir, f"{prefix}_{key}_{run_date}.pt")

    # ---- these are the only two lines that used to be hardcoded ----
    model = model_cls.build_model_from_params(cfg,
                                              best_params,
                                              device,
                                              verbose=verbose)
    loss_fn = model_cls.resolve_loss_fn(best_params)

    # everything below is unchanged
    if train_flag and (not force_retrain) and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    if not train_flag and not os.path.exists(model_filename):
        raise FileNotFoundError(
            f"No checkpoint found for {key}: {model_filename}")

    if not train_flag and os.path.exists(model_filename):
        state = torch.load(model_filename, map_location=device)
        model.load_state_dict(state)
        return model, model_filename, None

    # --- Train ---
    mbm.utils.seed_all(cfg.seed)

    ds_train = lstm_assets["ds_train"]
    train_idx = lstm_assets["train_idx"]
    val_idx = lstm_assets["val_idx"]

    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_train)

    train_dl, val_dl = ds_train_copy.make_loaders(
        train_idx=train_idx,
        val_idx=val_idx,
        batch_size_train=batch_size_train,
        batch_size_val=batch_size_val,
        seed=cfg.seed,
        fit_and_transform=True,
        shuffle_train=True,
        use_weighted_sampler=True,
        verbose=verbose,
    )

    if os.path.exists(model_filename):
        os.remove(model_filename)
        if verbose:
            print(f"Deleted existing model file: {model_filename}")

    history, best_val, best_state = model.train_loop(
        device=device,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=epochs,
        lr=best_params["lr"],
        weight_decay=best_params["weight_decay"],
        clip_val=1,
        sched_factor=0.5,
        sched_patience=6,
        sched_threshold=0.01,
        sched_threshold_mode="rel",
        sched_cooldown=1,
        sched_min_lr=1e-6,
        es_patience=15,
        es_min_delta=1e-4,
        log_every=5,
        verbose=verbose,
        save_best_path=model_filename,
        loss_fn=loss_fn,
    )

    if verbose:
        plot_history_lstm(history)

    state = torch.load(model_filename, map_location=device)
    model.load_state_dict(state)

    return model, model_filename, {"history": history, "best_val": best_val}

In [ ]:
def evaluate_one_model_pooled(
    cfg,
    model,
    device,
    region_assets: dict,  # {ds_train, train_idx, val_idx}
    ds_holdout,  # pristine holdout MBSequenceDataset
    ax=None,
    ax_xlim=(-8, 6),
    ax_ylim=(-8, 6),
    title=None,
    legend_fontsize=NATURE_SPECS["font_min_pt"],
    batch_size=128,
):
    # --- scaler donor from train ---
    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        region_assets["ds_train"])
    ds_train_copy.make_loaders(
        train_idx=region_assets["train_idx"],
        val_idx=region_assets["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )

    # --- test loader ---
    ds_holdout_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_holdout)
    test_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_holdout_copy,
        ds_train_copy,
        batch_size=batch_size,
        seed=cfg.seed,
    )

    # --- predict ---
    test_metrics, df_preds = model.evaluate_with_preds(device, test_dl,
                                                       ds_holdout_copy)
    df_preds = df_preds.dropna(subset=["target", "pred"])

    scores_annual, scores_winter = compute_seasonal_scores(df_preds,
                                                           target_col="target",
                                                           pred_col="pred")

    out = {
        "RMSE_annual":
        float(test_metrics.get("RMSE_annual", scores_annual["rmse"])),
        "RMSE_winter":
        float(test_metrics.get("RMSE_winter", scores_winter["rmse"])),
        "R2_annual":
        float(scores_annual["R2"]),
        "R2_winter":
        float(scores_winter["R2"]),
        "Bias_annual":
        float(scores_annual["Bias"]),
        "Bias_winter":
        float(scores_winter["Bias"]),
        "n_annual":
        int(scores_annual.get("n", 0)),
        "n_winter":
        int(scores_winter.get("n", 0)),
    }

    # --- plot ---
    created_fig = None
    if ax is None:
        created_fig = plt.figure(figsize=nature_figsize(cols=2, height_mm=120))
        ax = plt.subplot(1, 1, 1)

    pred_vs_truth_density(
        ax,
        df_preds,
        scores_annual,
        add_legend=False,
        palette=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
        ax_xlim=ax_xlim,
        ax_ylim=ax_ylim,
        s=50,
    )

    def _fmt(x):
        return "NA" if (x is None or
                        (isinstance(x, float) and np.isnan(x))) else f"{x:.2f}"

    legend_txt = "\n".join([
        rf"$\mathrm{{RMSE_a}}={_fmt(scores_annual['rmse'])},\ \mathrm{{RMSE_w}}={_fmt(scores_winter['rmse'])}$",
        rf"$\mathrm{{R^2_a}}={_fmt(scores_annual['R2'])},\ \mathrm{{R^2_w}}={_fmt(scores_winter['R2'])}$",
        rf"$\mathrm{{Bias_a}}={_fmt(scores_annual['Bias'])},\ \mathrm{{Bias_w}}={_fmt(scores_winter['Bias'])}$",
    ])
    ax.text(
        0.02,
        0.98,
        legend_txt,
        transform=ax.transAxes,
        va="top",
        fontsize=legend_fontsize,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.5),
    )

    if title:
        ax.set_title(title, fontsize=NATURE_SPECS["font_max_pt"])

    apply_nature_style(ax, fontsize=NATURE_SPECS["font_min_pt"], box=True)

    return out, df_preds, created_fig, ax

#### Models per region:

In [ ]:
gs_logs_dir = BASE_DIR / "logs/Transformer_GS"
logs_gs_transformer = pd.read_csv(
    os.path.join(gs_logs_dir, "transformer_gs_pooled_2026-05-20.csv"))

# best by validation loss
best_row = logs_gs_transformer.sort_values("valid_loss").iloc[0]
print("Best config:")
print(best_row[[
    "d_model", "nhead", "num_layers", "dim_feedforward", "dropout",
    "static_layers", "static_hidden", "static_dropout", "head_dropout", "lr",
    "weight_decay", "valid_loss", "val_rmse_a", "val_rmse_w"
]])

best_params_gs = {
    "Fm":
    int(best_row["Fm"]),
    "Fs":
    int(best_row["Fs"]),
    "d_model":
    int(best_row["d_model"]),
    "nhead":
    int(best_row["nhead"]),
    "num_layers":
    int(best_row["num_layers"]),
    "dim_feedforward":
    int(best_row["dim_feedforward"]),
    "dropout":
    float(best_row["dropout"]),
    "static_layers":
    int(best_row["static_layers"]),
    "static_hidden": (None if pd.isna(best_row["static_hidden"]) else int(
        best_row["static_hidden"])),
    "static_dropout": (None if pd.isna(best_row["static_dropout"]) else float(
        best_row["static_dropout"])),
    "head_dropout":
    float(best_row["head_dropout"]),
    "lr":
    float(best_row["lr"]),
    "weight_decay":
    float(best_row["weight_decay"]),
    "two_heads":
    False,
    "loss_spec":
    None,
    "T_max":
    32,
}

print("\nbest_params_gs:")
for k, v in best_params_gs.items():
    print(f"  {k}: {v}")

In [ ]:
MODEL_DATE = "2026-05-20"
TRAIN_FLAG = True

lstm_params = {
    'Fm': 8,
    'Fs': 3,
    'hidden_size': 128,
    'num_layers': 1,
    'bidirectional': False,
    'dropout': 0.0,
    'static_layers': 1,
    'static_hidden': 128,
    'static_dropout': 0.1,
    'lr': 0.001,
    'weight_decay': 1e-05,
    'two_heads': False,
    'head_dropout': 0.1,
    'loss_spec': None
}

transformer_params = best_params_gs

CONFIGS = {
    "lstm": (lstm_params, "lstm"),
    "transformer": (transformer_params, "transformer"),
}

all_metrics = {}
all_preds = {}
region_assets_cache = {}
models_cache = {}

for region in SOURCE_REGIONS:

    region_assets, region_holdout = build_pooled_assets_from_cache(
        res_xreg_by_source=res_xreg_by_source,
        splits=splits,
        source_regions=[region],
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        cfg=cfg,
        seed=cfg.seed,
        val_ratio=0.2,
        cache_dir=CACHE_DIR,
        force_recompute=True)
    region_assets_cache[region] = (region_assets, region_holdout)

    ds_train_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        region_assets["ds_train"])
    ds_train_copy.make_loaders(
        train_idx=region_assets["train_idx"],
        val_idx=region_assets["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )

    for model_type, (params, prefix_tag) in CONFIGS.items():

        print(f"\n{'='*60}")
        print(f"  {model_type.upper()} — {region}")
        print(f"{'='*60}")

        model, model_path, info = train_or_load_one_source_model(
            cfg=cfg,
            key=region,
            lstm_assets=region_assets,
            best_params=params,
            device=device,
            models_dir=BASE_DIR / "models/Transformer_exp",
            prefix=f"{prefix_tag}_{region}",
            train_flag=TRAIN_FLAG,
            force_retrain=True,
            epochs=150,
            date=MODEL_DATE,
            model_type=model_type,
        )
        models_cache[(region, model_type)] = model

        ds_holdout_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
            region_holdout[region])
        holdout_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
            ds_holdout_copy,
            ds_train_copy,
            batch_size=128,
            seed=cfg.seed,
        )

        metrics, df_preds = model.evaluate_with_preds(device, holdout_dl,
                                                      ds_holdout_copy)

        all_metrics[(region, model_type)] = metrics
        all_preds[(region, model_type)] = df_preds

        print(f"  RMSE annual : {metrics['RMSE_annual']:.3f}")
        print(f"  RMSE winter : {metrics['RMSE_winter']:.3f}")
        print(f"  R2 annual   : {metrics['R2_annual']:.3f}")
        print(f"  R2 winter   : {metrics['R2_winter']:.3f}")

# --- summary table ---
rows = []
for (region, model_type), m in all_metrics.items():
    rows.append({
        "region": region,
        "model": model_type,
        "RMSE_annual": round(m["RMSE_annual"], 3),
        "RMSE_winter": round(m["RMSE_winter"], 3),
        "R2_annual": round(m["R2_annual"], 3),
        "R2_winter": round(m["R2_winter"], 3),
    })
df_summary = pd.DataFrame(rows).set_index(["region", "model"]).sort_index()
print(df_summary)

#### Multi region training:

##### Zero shot:

In [ ]:
# ================================================================
# Multi-region pooled transformer training
# ================================================================

# --- 1. Build pooled assets (all three regions, holdouts excluded) ---
pooled_assets, ds_holdout_by_region = build_pooled_assets_from_cache(
    res_xreg_by_source=res_xreg_by_source,
    splits=splits,
    source_regions=SOURCE_REGIONS,  # ['CH', 'NOR', 'ISL']
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    cfg=cfg,
    seed=cfg.seed,
    val_ratio=0.2,
    cache_dir=CACHE_DIR,
    force_recompute=True,
)

# --- 2. Train pooled transformer ---
TRAIN_FLAG = True
model_pooled, path_pooled, info_pooled = train_or_load_one_source_model(
    cfg=cfg,
    key="CH_NOR_ISL",
    lstm_assets=pooled_assets,
    best_params=transformer_params,
    device=device,
    models_dir=BASE_DIR / "models/Transformer_exp",
    prefix="transformer_pooled",
    train_flag=TRAIN_FLAG,
    force_retrain=True,
    epochs=150,
    date=MODEL_DATE,
    model_type="transformer",
)

# --- 3. Scaler donor from pooled train ---
ds_pooled_scaler = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
    pooled_assets["ds_train"])
ds_pooled_scaler.make_loaders(
    train_idx=pooled_assets["train_idx"],
    val_idx=pooled_assets["val_idx"],
    fit_and_transform=True,
    seed=cfg.seed,
    verbose=False,
)

# --- 4. Evaluate zero-shot on each region's holdout ---
pooled_metrics = {}
pooled_preds = {}

for region in SOURCE_REGIONS:
    metrics, df_preds, _, _ = evaluate_one_model_pooled(
        cfg=cfg,
        model=model_pooled,
        device=device,
        region_assets={
            "ds_train": pooled_assets["ds_train"],
            "train_idx": pooled_assets["train_idx"],
            "val_idx": pooled_assets["val_idx"],
        },
        ds_holdout=ds_holdout_by_region[region],
        ax=None,
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        title=f"Pooled transformer — {region} holdout (zero-shot)",
    )
    pooled_metrics[region] = metrics
    pooled_preds[region] = df_preds

    print(f"\n{region} — pooled transformer (zero-shot)")
    print(f"  RMSE annual : {metrics['RMSE_annual']:.3f}")
    print(f"  RMSE winter : {metrics['RMSE_winter']:.3f}")
    print(f"  R2 annual   : {metrics['R2_annual']:.3f}")
    print(f"  R2 winter   : {metrics['R2_winter']:.3f}")

# --- 5. Extended summary table (per-region + pooled) ---
rows = []
for (region, model_type), m in all_metrics.items():  # from per-region loop
    rows.append({
        "region": region,
        "model": model_type,
        "RMSE_annual": round(m["RMSE_annual"], 3),
        "RMSE_winter": round(m["RMSE_winter"], 3),
        "R2_annual": round(m["R2_annual"], 3),
        "R2_winter": round(m["R2_winter"], 3),
    })
for region, m in pooled_metrics.items():
    rows.append({
        "region": region,
        "model": "transformer_pooled",
        "RMSE_annual": round(m["RMSE_annual"], 3),
        "RMSE_winter": round(m["RMSE_winter"], 3),
        "R2_annual": round(m["R2_annual"], 3),
        "R2_winter": round(m["R2_winter"], 3),
    })

df_summary = pd.DataFrame(rows).set_index(["region", "model"]).sort_index()
print("\n=== SUMMARY — per-region holdout ===")
print(df_summary)


##### Fine tuning:

In [ ]:
# ================================================================
# Fine-tuning pooled transformer on pool glaciers per region
# ================================================================

import copy


def finetune_pooled_transformer(
        cfg,
        model_pooled,
        region,
        splits,
        res_xreg_by_source,
        monthly_cols,
        static_cols,
        ds_pooled_scaler,
        device,
        models_dir,
        best_params,
        model_date,
        epochs=50,
        lr_factor=0.1,
        force_retrain=True,
        strategy="full",  # <-- new: "full" or "head_only"
):
    """Fine-tune the pooled transformer on one region's pool glaciers."""
    assert strategy in ("full", "head_only"), \
        f"strategy must be 'full' or 'head_only', got '{strategy}'"

    os.makedirs(models_dir, exist_ok=True)
    model_filename = os.path.join(
        models_dir,
        f"transformer_pooled_ft_{strategy}_{region}_{model_date}.pt")

    # --- build fine-tune dataset (unchanged) ---
    pool_glaciers = splits[region]['pool_glaciers']
    df_loss = res_xreg_by_source[region]['data_monthly']
    df_full = res_xreg_by_source[region]['data_monthly_aug']
    months_head_pad = res_xreg_by_source[region]['months_head_pad']
    months_tail_pad = res_xreg_by_source[region]['months_tail_pad']

    df_loss_ft = df_loss[df_loss['GLACIER'].isin(pool_glaciers)]
    df_full_ft = df_full[df_full['GLACIER'].isin(pool_glaciers)]

    print(
        f"\n{region} fine-tune [{strategy}]: "
        f"{df_loss_ft['GLACIER'].nunique()} glaciers, {len(df_loss_ft)} rows")

    ds_ft = build_combined_LSTM_dataset(
        df_loss=df_loss_ft,
        df_full=df_full_ft,
        monthly_cols=monthly_cols,
        static_cols=static_cols,
        months_head_pad=months_head_pad,
        months_tail_pad=months_tail_pad,
        normalize_target=True,
        expect_target=True,
        show_progress=False,
    )

    train_idx_ft, val_idx_ft = mbm.data_processing.MBSequenceDataset.split_indices(
        len(ds_ft), val_ratio=0.2, seed=cfg.seed)

    ds_ft_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_ft)
    ds_ft_copy.set_scalers_from(ds_pooled_scaler)
    ds_ft_copy.transform_inplace()

    train_dl, val_dl = ds_ft_copy.make_loaders(
        train_idx=train_idx_ft,
        val_idx=val_idx_ft,
        fit_and_transform=False,
        seed=cfg.seed,
        use_weighted_sampler=True,
        verbose=True,
    )

    model_ft = copy.deepcopy(model_pooled)

    if not force_retrain and os.path.exists(model_filename):
        print(f"Loading fine-tuned model [{strategy}] from {model_filename}")
        state = torch.load(model_filename, map_location=device)
        model_ft.load_state_dict(state)
        return model_ft, model_filename

    # --- freeze / unfreeze based on strategy ---
    if strategy == "head_only":
        # freeze everything
        for param in model_ft.parameters():
            param.requires_grad = False
        # unfreeze head(s) only
        if model_ft.two_heads:
            for param in model_ft.head_w.parameters():
                param.requires_grad = True
            for param in model_ft.head_a.parameters():
                param.requires_grad = True
        else:
            for param in model_ft.head.parameters():
                param.requires_grad = True

        n_trainable = sum(p.numel() for p in model_ft.parameters()
                          if p.requires_grad)
        n_total = sum(p.numel() for p in model_ft.parameters())
        print(f"  head_only: {n_trainable} / {n_total} params trainable")

    # optimizer only sees trainable params
    optimizer = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model_ft.parameters()),
        lr=best_params['lr'] * lr_factor,
        weight_decay=best_params['weight_decay'],
    )
    loss_fn = mbm.models.Transformer_MB.resolve_loss_fn(best_params)

    history, best_val, _ = model_ft.train_loop(
        device=device,
        train_dl=train_dl,
        val_dl=val_dl,
        epochs=epochs,
        optimizer=optimizer,
        loss_fn=loss_fn,
        es_patience=10,
        es_min_delta=1e-4,
        sched_factor=0.5,
        sched_patience=4,
        sched_threshold=0.01,
        sched_threshold_mode="rel",
        sched_cooldown=1,
        sched_min_lr=1e-7,
        log_every=5,
        verbose=True,
        save_best_path=model_filename,
    )

    plot_history_lstm(history)

    state = torch.load(model_filename, map_location=device)
    model_ft.load_state_dict(state)

    return model_ft, model_filename

In [ ]:
# --- run fine-tuning for each region ---
models_ft = {}
TRAIN_FLAG = True
for strategy in ["full", "head_only"]:
    models_ft[strategy] = {}
    for region in SOURCE_REGIONS:
        model_ft, path_ft = finetune_pooled_transformer(
            cfg=cfg,
            model_pooled=model_pooled,
            region=region,
            splits=splits,
            res_xreg_by_source=res_xreg_by_source,
            monthly_cols=MONTHLY_COLS,
            static_cols=STATIC_COLS,
            ds_pooled_scaler=ds_pooled_scaler,
            device=device,
            models_dir=BASE_DIR / "models/Transformer_exp",
            best_params=transformer_params,
            model_date=MODEL_DATE,
            epochs=50,
            lr_factor=0.1,
            force_retrain=TRAIN_FLAG,
            strategy=strategy,
        )
        models_ft[strategy][region] = model_ft
        print(f"{region} [{strategy}] -> {path_ft}")

# --- evaluate fine-tuned models on holdout ---
ft_metrics = {strategy: {} for strategy in models_ft}
ft_preds = {strategy: {} for strategy in models_ft}

for strategy, strategy_models in models_ft.items():
    print(f"\n{'='*60}")
    print(f"Evaluating strategy: {strategy}")
    for region in SOURCE_REGIONS:
        metrics, df_preds, _, _ = evaluate_one_model_pooled(
            cfg=cfg,
            model=strategy_models[region],
            device=device,
            region_assets={
                "ds_train": pooled_assets["ds_train"],
                "train_idx": pooled_assets["train_idx"],
                "val_idx": pooled_assets["val_idx"],
            },
            ds_holdout=ds_holdout_by_region[region],
            ax=None,
            ax_xlim=(-16, 10),
            ax_ylim=(-16, 10),
            title=f"Pooled transformer [{strategy}] — {region} holdout",
        )
        ft_metrics[strategy][region] = metrics
        ft_preds[strategy][region] = df_preds

        print(f"  {region} [{strategy}]  "
              f"RMSE_a={metrics['RMSE_annual']:.3f}  "
              f"RMSE_w={metrics['RMSE_winter']:.3f}  "
              f"R2_a={metrics['R2_annual']:.3f}  "
              f"R2_w={metrics['R2_winter']:.3f}")

# --- full summary table ---
rows = []

# per-region baselines
for (region, model_type), m in all_metrics.items():
    rows.append({
        "region": region,
        "model": model_type,
        **{
            k: round(m[k], 3)
            for k in ["RMSE_annual", "RMSE_winter", "R2_annual", "R2_winter"]
        }
    })

# pooled zero-shot
for region, m in pooled_metrics.items():
    rows.append({
        "region": region,
        "model": "transformer_pooled",
        **{
            k: round(m[k], 3)
            for k in ["RMSE_annual", "RMSE_winter", "R2_annual", "R2_winter"]
        }
    })

# fine-tuned strategies
for strategy, region_metrics in ft_metrics.items():
    for region, m in region_metrics.items():
        rows.append({
            "region": region,
            "model": f"transformer_pooled_ft_{strategy}",
            **{
                k: round(m[k], 3)
                for k in [
                    "RMSE_annual", "RMSE_winter", "R2_annual", "R2_winter"
                ]
            }
        })

df_summary = pd.DataFrame(rows).set_index(["region", "model"]).sort_index()
print("\n=== FULL SUMMARY ===")
print(df_summary)

In [ ]:
# --- final grid: 4 columns ---
COL_CONFIGS = [
    ("lstm", "LSTM\n(per-region)", lambda r: region_assets_cache[r][0],
     lambda r: region_assets_cache[r][1][r], lambda r: models_cache[
         (r, "lstm")]),
    ("transformer", "Transformer\n(per-region)",
     lambda r: region_assets_cache[r][0],
     lambda r: region_assets_cache[r][1][r], lambda r: models_cache[
         (r, "transformer")]),
    ("transformer_pooled", "Transformer\n(pooled)", lambda r: pooled_assets,
     lambda r: ds_holdout_by_region[r], lambda r: model_pooled),
    ("ft_head_only", "Transformer\n(pooled+FT head)", lambda r: pooled_assets,
     lambda r: ds_holdout_by_region[r], lambda r: models_ft["head_only"][r]),
]

nrows = len(SOURCE_REGIONS)
ncols = len(COL_CONFIGS)

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 5, nrows * 5),
    sharex=True,
    sharey=True,
)

for row_idx, region in enumerate(SOURCE_REGIONS):
    for col_idx, (_, col_label, get_assets, get_holdout,
                  get_model) in enumerate(COL_CONFIGS):
        ax = axes[row_idx][col_idx]

        metrics, df_preds, _, _ = evaluate_one_model_pooled(
            cfg=cfg,
            model=get_model(region),
            device=device,
            region_assets=get_assets(region),
            ds_holdout=get_holdout(region),
            ax=ax,
            ax_xlim=(-16, 10),
            ax_ylim=(-16, 10),
            title=None,
            legend_fontsize=NATURE_SPECS["font_max_pt"],
        )

        if col_idx == 0:
            ax.set_ylabel(f"{region}\nModeled PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")

        if row_idx == 0:
            ax.set_title(col_label, fontsize=NATURE_SPECS["font_max_pt"] + 2)

        if row_idx == nrows - 1:
            ax.set_xlabel("Observed PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_xlabel("")

        leg = ax.get_legend()
        if leg is not None:
            leg.remove()

legend_handles = [
    mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
    mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    "Holdout evaluation — LSTM | Transformer per-region | Pooled | Pooled + FT",
    fontsize=NATURE_SPECS["font_max_pt"] + 3,
    y=1.01,
)
fig.tight_layout()
plt.show()

### Application:

In [ ]:
# Target region:
TGT_REGION = ['SJM']

# Step 1: only build monthly cache for these three
run_flag_by_code = {
    'SJM': False,
}

monthly_cache_tgt = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=vois_climate,
    vois_topographical=vois_topographical,
    region_codes=TGT_REGION,
    run_flag_by_code=run_flag_by_code,
)

# Step 2: no XREG_PAIRS needed — just get the train/test split per region
# res_xreg_by_source[region] gives you df_train, df_test, df_train_aug, df_test_aug
res_xreg_by_tgt = {
    region: monthly_cache_tgt[region]  # already has the train/test split
    for region in TGT_REGION
}

In [ ]:
# --- build SJM dataset (all glaciers, no holdout split) ---
ds_sjm = build_combined_LSTM_dataset(
    df_loss=res_xreg_by_tgt['SJM']['data_monthly'],
    df_full=res_xreg_by_tgt['SJM']['data_monthly_aug'],
    monthly_cols=MONTHLY_COLS,
    static_cols=STATIC_COLS,
    months_head_pad=res_xreg_by_tgt['SJM']['months_head_pad'],
    months_tail_pad=res_xreg_by_tgt['SJM']['months_tail_pad'],
    normalize_target=True,
    expect_target=True,
    show_progress=True,
)
print(f"SJM dataset: {len(ds_sjm)} sequences  "
      f"({ds_sjm.iw.sum().item()} winter | {ds_sjm.ia.sum().item()} annual)")


def evaluate_on_sjm(model, region_assets_for_scaler, label):
    """Evaluate any model zero-shot on the full SJM dataset."""
    # scaler donor — use the training assets of whatever model we're evaluating
    ds_scaler_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        region_assets_for_scaler["ds_train"])
    ds_scaler_copy.make_loaders(
        train_idx=region_assets_for_scaler["train_idx"],
        val_idx=region_assets_for_scaler["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )

    ds_sjm_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_sjm)
    sjm_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_sjm_copy,
        ds_scaler_copy,
        batch_size=128,
        seed=cfg.seed,
    )

    metrics, df_preds = model.evaluate_with_preds(device, sjm_dl, ds_sjm_copy)
    df_preds = df_preds.dropna(subset=["target", "pred"])

    print(f"  {label:45s}  "
          f"RMSE_a={metrics['RMSE_annual']:.3f}  "
          f"RMSE_w={metrics['RMSE_winter']:.3f}  "
          f"R2_a={metrics['R2_annual']:.3f}  "
          f"R2_w={metrics['R2_winter']:.3f}")

    return metrics, df_preds


# --- evaluate all models on SJM ---
print(f"\n{'='*60}")
print("Zero-shot evaluation on SJM (Svalbard)")
print(f"{'='*60}")

sjm_metrics = {}
sjm_preds = {}

# per-region baselines
for region in SOURCE_REGIONS:
    region_assets, _ = region_assets_cache[region]
    for model_type in ["lstm", "transformer"]:
        label = f"{model_type} ({region})"
        m, p = evaluate_on_sjm(
            model=models_cache[(region, model_type)],
            region_assets_for_scaler=region_assets,
            label=label,
        )
        sjm_metrics[label] = m
        sjm_preds[label] = p

# pooled transformer (zero-shot)
m, p = evaluate_on_sjm(
    model=model_pooled,
    region_assets_for_scaler=pooled_assets,
    label="transformer (pooled, zero-shot)",
)
sjm_metrics["transformer (pooled, zero-shot)"] = m
sjm_preds["transformer (pooled, zero-shot)"] = p

# fine-tuned strategies
for strategy, strategy_models in models_ft.items():
    for region in SOURCE_REGIONS:
        label = f"transformer (pooled+FT {strategy}, trained on {region} pool)"
        m, p = evaluate_on_sjm(
            model=strategy_models[region],
            region_assets_for_scaler=pooled_assets,
            label=label,
        )
        sjm_metrics[label] = m
        sjm_preds[label] = p

# --- SJM summary table ---
rows = [{
    "model": label,
    **{
        k: round(m[k], 3)
        for k in ["RMSE_annual", "RMSE_winter", "R2_annual", "R2_winter"]
    }
} for label, m in sjm_metrics.items()]
df_sjm_summary = pd.DataFrame(rows).set_index("model")
print("\n=== SJM summary ===")
print(df_sjm_summary)

In [ ]:
SJM_PLOT_CONFIGS = [
    # per-region (NOR most relevant climatically)
    ("LSTM (NOR)", models_cache[("NOR", "lstm")], region_assets_cache["NOR"][0]
     ),
    ("LSTM (CH)", models_cache[("CH", "lstm")], region_assets_cache["CH"][0]),
    ("LSTM (ISL)", models_cache[("ISL", "lstm")],
     region_assets_cache["ISL"][0]),
    ("Transformer (NOR)", models_cache[("NOR", "transformer")],
     region_assets_cache["NOR"][0]),
    ("Transformer (CH)", models_cache[("CH", "transformer")],
     region_assets_cache["CH"][0]),
    ("Transformer (ISL)", models_cache[("ISL", "transformer")],
     region_assets_cache["ISL"][0]),
    # pooled
    ("Transformer (pooled)", model_pooled, pooled_assets),
    # best FT per strategy — NOR pool most relevant for SJM
    *[(f"Transformer (pooled+FT {strategy}, NOR pool)",
       models_ft[strategy]["NOR"], pooled_assets) for strategy in models_ft],
]

ncols = 3  # one col per source region for per-region models; adjust freely
nrows = int(np.ceil(len(SJM_PLOT_CONFIGS) / ncols))

fig, axes = plt.subplots(
    nrows,
    ncols,
    figsize=(ncols * 5, nrows * 5),
    sharex=True,
    sharey=True,
)
axes = np.array(axes).reshape(-1)

for idx, (label, model, region_assets) in enumerate(SJM_PLOT_CONFIGS):
    ax = axes[idx]

    # scaler donor
    ds_scaler_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        region_assets["ds_train"])
    ds_scaler_copy.make_loaders(
        train_idx=region_assets["train_idx"],
        val_idx=region_assets["val_idx"],
        fit_and_transform=True,
        seed=cfg.seed,
        verbose=False,
    )

    ds_sjm_copy = mbm.data_processing.MBSequenceDataset._clone_untransformed_dataset(
        ds_sjm)
    sjm_dl = mbm.data_processing.MBSequenceDataset.make_test_loader(
        ds_sjm_copy,
        ds_scaler_copy,
        batch_size=128,
        seed=cfg.seed,
    )

    test_metrics, df_preds = model.evaluate_with_preds(device, sjm_dl,
                                                       ds_sjm_copy)
    df_preds = df_preds.dropna(subset=["target", "pred"])

    scores_annual, scores_winter = compute_seasonal_scores(df_preds,
                                                           target_col="target",
                                                           pred_col="pred")

    pred_vs_truth_density(
        ax,
        df_preds,
        scores_annual,
        add_legend=False,
        palette=[mbm.plots.COLOR_ANNUAL, mbm.plots.COLOR_WINTER],
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        s=50,
    )

    def _fmt(x):
        return "NA" if (x is None or
                        (isinstance(x, float) and np.isnan(x))) else f"{x:.2f}"

    legend_txt = "\n".join([
        rf"$\mathrm{{RMSE_a}}={_fmt(scores_annual['rmse'])},\ \mathrm{{RMSE_w}}={_fmt(scores_winter['rmse'])}$",
        rf"$\mathrm{{R^2_a}}={_fmt(scores_annual['R2'])},\ \mathrm{{R^2_w}}={_fmt(scores_winter['R2'])}$",
        rf"$\mathrm{{Bias_a}}={_fmt(scores_annual['Bias'])},\ \mathrm{{Bias_w}}={_fmt(scores_winter['Bias'])}$",
    ])

    apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
    ax.set_title(label, fontsize=NATURE_SPECS["font_max_pt"] + 5)
    ax.text(
        0.02,
        0.98,
        legend_txt,
        transform=ax.transAxes,
        va="top",
        fontsize=NATURE_SPECS["font_max_pt"],
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.5),
    )

    col = idx % ncols
    row = idx // ncols
    ax.set_xlabel("Observed PMB (m w.e.)" if row == nrows - 1 else "",
                  fontsize=NATURE_SPECS["font_max_pt"])
    ax.set_ylabel("Modeled PMB (m w.e.)" if col == 0 else "",
                  fontsize=NATURE_SPECS["font_max_pt"])
    leg = ax.get_legend()
    if leg is not None:
        leg.remove()

# turn off unused axes
for j in range(len(SJM_PLOT_CONFIGS), len(axes)):
    axes[j].axis("off")

legend_handles = [
    mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
    mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
]
fig.legend(
    handles=legend_handles,
    loc="lower center",
    bbox_to_anchor=(0.5, -0.02),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    "Zero-shot generalisation to Svalbard (SJM)",
    fontsize=NATURE_SPECS["font_max_pt"] + 3,
    y=1.01,
)
fig.tight_layout()
plt.show()